# 🧠 InceptionV3 Hap Sınıflandırma — Eğitim & Kayıt
**Amaç:** InceptionV3 modelini eğit, `Hap_Modelleri_V3` klasörüne kaydet.
**Giriş boyutu:** 299×299 | **Preprocessing:** [-1, 1] normalize

In [ ]:
# ── HÜCRE 1: Drive Bağlantısı ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive bağlandı!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive bağlandı!


In [ ]:
# ── HÜCRE 2: Kütüphaneler ──────────────────────────────────────────────────
import os, cv2, numpy as np, random, warnings
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

warnings.filterwarnings('ignore')
print('✅ Kütüphaneler yüklendi!')

✅ Kütüphaneler yüklendi!


In [ ]:
# ── HÜCRE 3: Veri Yolu ve Sınıflar ────────────────────────────────────────
veri_yolu = "/content/drive/MyDrive/YapayZeka_Hap_Projesi/archive/Drug Vision/Data Combined"
siniflar  = sorted([s for s in os.listdir(veri_yolu)
                    if os.path.isdir(os.path.join(veri_yolu, s))])

print(f"✅ Toplam {len(siniflar)} İlaç Sınıfı:")
for i, s in enumerate(siniflar):
    print(f"  [{i}] {s}")

✅ Toplam 10 İlaç Sınıfı:
  [0] Alaxan
  [1] Bactidol
  [2] Bioflu
  [3] Biogesic
  [4] DayZinc
  [5] Decolgen
  [6] Fish Oil
  [7] Kremil S
  [8] Medicol
  [9] Neozep


In [ ]:
# ── HÜCRE 4: Veri Yükleme ──────────────────────────────────────────────────
# InceptionV3 minimum 75x75, önerilen 299x299.
# preprocess_input sonraya bırakılıyor — uint8 RGB olarak saklanır.
# Augmentation YOK.
BOYUT = (299, 299)
print('📂 Veriler yükleniyor... (299x299 — biraz sürebilir)')

X, y = [], []
for sinif_adi in siniflar:
    yol = os.path.join(veri_yolu, sinif_adi)
    for resim_adi in os.listdir(yol):
        r = cv2.imread(os.path.join(yol, resim_adi))
        if r is None:
            continue
        r = cv2.resize(r, BOYUT)
        X.append(cv2.cvtColor(r, cv2.COLOR_BGR2RGB))
        y.append(sinif_adi)

X = np.array(X, dtype='float32')
y = np.array(y)
print(f'✅ Toplam görüntü: {len(y)} adet.')

📂 Veriler yükleniyor... (299x299 — biraz sürebilir)
✅ Toplam görüntü: 10000 adet.


In [ ]:
# ── HÜCRE 5: InceptionV3 Preprocessing + Train/Test Bölme ─────────────────
# preprocess_input: [0,255] → [-1, 1] aralığına taşır (InceptionV3'e özgü).
# ResNet'in mean subtraction'ından farklı — karıştırılmamalı.

SEED = random.randint(0, 9999)
print(f'🎲 Seed: {SEED}')

le = LabelEncoder()
le.fit(siniflar)
y_num = le.transform(y)

X_train_raw, X_test_raw, y_train_num, y_test_num = train_test_split(
    X, y_num, test_size=0.2, random_state=SEED, stratify=y_num
)

# InceptionV3'e özgü normalizasyon
X_train = preprocess_input(X_train_raw.copy())
X_test  = preprocess_input(X_test_raw.copy())

y_train_hot = tf.keras.utils.to_categorical(y_train_num, num_classes=len(siniflar))
y_test_hot  = tf.keras.utils.to_categorical(y_test_num,  num_classes=len(siniflar))

print(f'✅ Eğitim: {len(X_train)} | Test: {len(X_test)}')

🎲 Seed: 3846
✅ Eğitim: 8000 | Test: 2000


In [ ]:
# ── HÜCRE 6: tf.data Pipeline ──────────────────────────────────────────────
BATCH = 32

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train_hot)) \
    .shuffle(2000, seed=SEED).batch(BATCH).prefetch(tf.data.AUTOTUNE)
val_ds   = tf.data.Dataset.from_tensor_slices((X_test, y_test_hot)) \
    .batch(BATCH).prefetch(tf.data.AUTOTUNE)

print(f'✅ Pipeline hazır. Batch boyutu: {BATCH}')

✅ Pipeline hazır. Batch boyutu: 32


In [ ]:
# ── HÜCRE 7: InceptionV3 Model Mimarisi ───────────────────────────────────
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False  # İlk aşamada tamamen dondur

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(len(siniflar), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'✅ InceptionV3 modeli hazır.')
print(f'   Eğitilebilir parametre: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

✅ InceptionV3 modeli hazır.
   Eğitilebilir parametre: 1,187,082


In [ ]:
# ── HÜCRE 8: Aşama 1 — Sadece Baş Katmanları Eğit ─────────────────────────
kayit_yolu = '/content/drive/MyDrive/Hap_Modelleri_V3/inceptionv3_hap_modeli.h5'
os.makedirs(os.path.dirname(kayit_yolu), exist_ok=True)

callbacks_1 = [
    EarlyStopping(patience=5, restore_best_weights=True,
                  monitor='val_accuracy', verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-6,
                     monitor='val_loss', verbose=1),
    ModelCheckpoint(kayit_yolu, save_best_only=True,
                    monitor='val_accuracy', verbose=1)
]

print('🚀 Aşama 1 — Transfer Learning Başlıyor...')
history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_1,
    verbose=1
)
print(f'✅ Aşama 1 tamamlandı! En iyi val_accuracy: %{max(history_1.history["val_accuracy"])*100:.2f}')

NameError: name 'os' is not defined

In [ ]:
# ── HÜCRE 9: Aşama 2 — Fine-Tuning (Son 50 Katman) ────────────────────────
# InceptionV3'ün son inception bloklarını çok düşük LR ile açıyoruz.

base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_2 = [
    EarlyStopping(patience=4, restore_best_weights=True,
                  monitor='val_accuracy', verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=2, min_lr=1e-7,
                     monitor='val_loss', verbose=1),
    ModelCheckpoint(kayit_yolu, save_best_only=True,
                    monitor='val_accuracy', verbose=1)
]

print('🔧 Aşama 2 — Fine-Tuning Başlıyor...')
history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_2,
    verbose=1
)
print(f'✅ Fine-Tuning tamamlandı! En iyi val_accuracy: %{max(history_2.history["val_accuracy"])*100:.2f}')

In [ ]:
# ── HÜCRE 10: Final Değerlendirme ─────────────────────────────────────────
from tensorflow.keras.models import load_model

best_model = load_model(kayit_yolu)
loss, acc  = best_model.evaluate(val_ds, verbose=0)

print('\n' + '='*50)
print(f'  ✅ InceptionV3 Final Test Doğruluğu: %{acc*100:.2f}')
print(f'  📍 Model kaydedildi: {kayit_yolu}')
print('='*50)